# <font color="#ff00ff">Agentic RAG System Using ReAct Paradigm</font>

<font color="blue"> Learn how to build an **Agentic RAG System** to chat with Financial Reports such as 10k using **ReAct Agent** </font>

* Built a ReAct agent using LlamaIndex’s ReActAgent to answer financial comparison questions using retrieval + reasoning.

* Two Query Engines are used as tools: one for Apple’s fiscal 2025 10-K, the other for Nvidia’s — each tied to their respective document indexes.

* Agent uses OpenAI’s gpt-4.1-mini, with tool-calling enabled via the QueryEngineTool, allowing dynamic retrieval before reasoning.

* Asked complex financial questions, like comparing fiscal 2025 vs. 2024 revenues, and Apple’s vs. Nvidia’s revenues.

* Agent follows ReAct paradigm: identifies tool use if needed, fetches relevant context, and responds with calculated insights (e.g., % increase, ratio comparisons).

# Technical Stack

* LlamaIndex:  https://docs.llamaindex.ai/en/stable/examples/agent/agent_workflow_basic/

# <font color="blue"> Agentic RAG </font>

* Before starting building this agentic RAG system with a ReAct agent, let's have a look on what does mean ReAct Paradigm:

✅ ReAct means <font color="blue"> Reasoning + Acting</font>.

>> The LLM will generate both a reasoning traces, called also internal **Thought**, and **Actions**.

>> After taking the action, the LMM will generate what we call an **Observation**, which is the result of the action.

>> Then the **Thought + Action + Observation** will constitue what we call **Context**.

>> Based on this context the agent will end its execution if the user request is answered or enter in a new step: **Thought + Action + Observation**

In [1]:
from IPython.display import Image, display
display(Image(url="https://raw.githubusercontent.com/hananedupouy/LLMs-in-Finance/main/IMA_livre_blanc/images/reasoning_pattern_ReAct.png", width=900))

⏰ Let's start building our RAG ReAct agent

We'll use the OpenAI LLMs and embedding model. the latest is used after retrieving data from pdfs and storing then in a vectorstore database.

In [2]:
%pip install -q "llama-index>=0.14,<0.15" llama-index-llms-openai llama-index-embeddings-openai beautifulsoup4 requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
try:
    from google.colab import userdata  # Colab: add OPENAI_API_KEY in the Secrets panel
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except ImportError:
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]  # local: export it or use a .env
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # so every OpenAI(...) client finds it

In [4]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

MODEL = "gpt-4.1-mini"               # swap for gpt-5.1 for stronger reasoning
EMBED_MODEL = "text-embedding-3-small"
Settings.llm = OpenAI(model=MODEL)
Settings.embed_model = OpenAIEmbedding(model=EMBED_MODEL)

We'll load the latest 10-K reports for Apple and Nvidia straight from SEC EDGAR (the official source), so the comparison is fiscal 2025 vs fiscal 2024:

Apple 10-K fiscal 2025 (year ended September 27, 2025, filed October 31, 2025): https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm

Nvidia 10-K fiscal 2025 (year ended January 26, 2025, filed February 26, 2025): https://www.sec.gov/Archives/edgar/data/1045810/000104581025000023/nvda-20250126.htm

EDGAR filings are HTML (inline XBRL). We download each one with a proper User-Agent (the SEC requires it), strip the tags, and wrap the text in a LlamaIndex `Document`. A PDF from the investor-relations site works the same way with `SimpleDirectoryReader(input_files=[...])`.

In [5]:
import requests
from bs4 import BeautifulSoup
from llama_index.core import Document

HEADERS = {"User-Agent": "Hanane Dupouy hanane@ai-agent-in-finance.com"}  # the SEC asks for a name + contact in the UA

FILINGS = {
    "apple":  ("https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm",   "Apple 10-K FY2025"),
    "nvidia": ("https://www.sec.gov/Archives/edgar/data/1045810/000104581025000023/nvda-20250126.htm", "Nvidia 10-K FY2025"),
}

def load_10k(url: str, label: str) -> list[Document]:
    """Download an EDGAR 10-K (inline XBRL HTML) and return it as one LlamaIndex Document."""
    html = requests.get(url, headers=HEADERS, timeout=60).text
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "ix:header"]):   # ix:header = hidden XBRL metadata, not prose
        tag.decompose()
    text = "\n".join(line.strip() for line in soup.get_text("\n").splitlines() if line.strip())
    print(f"{label}: {len(text):,} characters of text")
    return [Document(text=text, metadata={"source": url, "filing": label})]

apple_docs = load_10k(*FILINGS["apple"])
nvidia_docs = load_10k(*FILINGS["nvidia"])

Apple 10-K FY2025: 206,827 characters of text
Nvidia 10-K FY2025: 348,385 characters of text


In [6]:
## If you prefer PDFs from the investor-relations sites, download them and use:
# from llama_index.core import SimpleDirectoryReader
# apple_docs = SimpleDirectoryReader(input_files=["./apple_10k_fy2025.pdf"]).load_data()
# (needs: %pip install llama-index-readers-file)

## Documents Storing and Query Engine

If documents are already stored:

In [7]:
from llama_index.core import StorageContext, load_index_from_storage

try:
    storage_context = StorageContext.from_defaults(
        persist_dir="./data_storage/apple_fy2025"
    )
    apple_index = load_index_from_storage(storage_context)

    storage_context = StorageContext.from_defaults(
        persist_dir="./data_storage/nvidia_fy2025"
    )
    nvidia_index = load_index_from_storage(storage_context)

    index_loaded = True
except:
    index_loaded = False

In [8]:
index_loaded

True

We load the filings into a VectorStoreIndex if they are not already stored:
You will see a new folder named data_storage, containing two subfolders: apple_fy2025 and nvidia_fy2025, with the index files.

In [9]:
from llama_index.core import VectorStoreIndex

if not index_loaded:
    # build one index per filing (chunk -> embed -> vector store)
    apple_index = VectorStoreIndex.from_documents(apple_docs)
    nvidia_index = VectorStoreIndex.from_documents(nvidia_docs)
    # persist, one folder per filing and fiscal year
    apple_index.storage_context.persist(persist_dir="./data_storage/apple_fy2025")
    nvidia_index.storage_context.persist(persist_dir="./data_storage/nvidia_fy2025")

We build a query engine on the top of the vectorestore index for each 10K report:

* This query engine allows retrieve the most relevant chunks related to the user query.

* We have then 2 query engines : **apple_engine** and **nvidia_engine**.

In [10]:
apple_engine = apple_index.as_query_engine(similarity_top_k=3)
nvidia_engine = nvidia_index.as_query_engine(similarity_top_k=3)

## Tools on the top of Query Engine

* The query engines are wrapped as tools that the ReAct agent can invoke during its reasoning process.

* 2 tools are created then "**apple_10k**" and "**nvidia_10k**".

In [11]:
from llama_index.core.tools import QueryEngineTool

query_engine_tools = [
    QueryEngineTool.from_defaults(
        query_engine=apple_engine,
        name="apple_10k",
        description=(
              "Delivers insights and information from Apple's fiscal 2025 10-K (year ended September 27, 2025), including fiscal 2024 comparatives. "
              "You'll be provided with a detailed, plain text question to obtain the most relevant and precise responses"
        ),
    ),
    QueryEngineTool.from_defaults(
        query_engine=nvidia_engine,
        name="nvidia_10k",
        description=(
              "Delivers insights and information from Nvidia's fiscal 2025 10-K (year ended January 26, 2025), including fiscal 2024 comparatives. "
              "You'll be provided with a detailed, plain text question to obtain the most relevant and precise responses"
        ),
    ),
]

## ReAct Agent

We'll be using the built-in ReAct Agent:

* Both tools will be added to the ReAct Agent.

* Based on the user request, the ReAct agent will choose one of the tools or both tools to retrieve the adequate information from the reports.

* We also added a **context** to the agent, so it can memorize the past conversations.

In [12]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context

agent = ReActAgent(
    tools=query_engine_tools,
    llm=Settings.llm,
)
ctx = Context(agent)

When asking: "What was Nvidia's revenue in fiscal 2025? Compare it to fiscal 2024."

➡ The agent understands that its needs to retrieve data from a tool: **Thought**

➡ The agent defines an **Action** which will call the nvidia_10k

➡ The agent calls "nvidia_10k" with the argument {"input": user query}

➡ We get the **observation** (= result) directly in the answer.

In [13]:
from llama_index.core.agent.workflow import ToolCallResult, AgentStream

handler = agent.run("What was Nvidia's revenue in fiscal 2025? Compare it to fiscal 2024.", ctx=ctx)
async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)
response = await handler

Thought: The user is asking for Nvidia's revenue in fiscal 2025 and a comparison to fiscal 2024. I need to use the nvidia_10k tool to get the most accurate and detailed information from Nvidia's fiscal 2025 10-K report.
Action: nvidia_10k
Action Input: {"input": "What was Nvidia's revenue in fiscal 2025? Please provide a comparison to fiscal 2024."}Thought: I have the information about Nvidia's revenue for fiscal 2025 and the comparison to fiscal 2024. I can now provide the answer in the user's language, which is English.

Answer: Nvidia's revenue for fiscal year 2025 was $130,497 million. This represents a significant increase of 114% compared to the fiscal year 2024 revenue, which was $60,922 million.

For this question, we are asking the agent to compare Apple's and Nvidia's fiscal 2025 revenues and give the ratio:

As you can see in the results, only **apple_10k** tool was called, because the agent has already in its context Nvidia's fiscal 2025 revenue.

The agent did a comparison between both values and even compute the ratio.

In [14]:
handler = agent.run("What was Apple's revenue in fiscal 2025? Compare it to Nvidia's fiscal 2025 revenue: how many times larger is it?", ctx=ctx)
async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)
response = await handler

Thought: The user wants to know Apple's revenue in fiscal 2025 and a comparison to Nvidia's fiscal 2025 revenue in terms of how many times larger Apple's revenue is. I will use the apple_10k tool to get Apple's fiscal 2025 revenue first.
Action: apple_10k
Action Input: {"input": "What was Apple's revenue in fiscal 2025? Please provide the revenue figure."}Thought: I have Apple's revenue for fiscal 2025 as $416,161 million and Nvidia's revenue for fiscal 2025 as $130,497 million. I need to calculate how many times larger Apple's revenue is compared to Nvidia's revenue.
Answer: Apple's revenue in fiscal 2025 was $416,161 million. Compared to Nvidia's fiscal 2025 revenue of $130,497 million, Apple's revenue is approximately 3.19 times larger.

# AI Agent In Finance Course Online

Sign up for my training, AI Agents in Finance (self-paced foundations, then the flagship cohort):

https://ai-agent-in-finance.com/training/

You'll find the detailed syllabus there.

# Readings

1- **Agentic AI Systems Applied to tasks in Financial Services: Modeling and model risk management crews**
https://arxiv.org/abs/2502.05439

2- AI Agents vs. Agentic AI: A Conceptual Taxonomy, Applications and Challenges
https://arxiv.org/pdf/2505.10468


3- AI Agents: Evolution, Architecture, and Real-World Applications
https://arxiv.org/html/2503.12687v1


4- LlamaIndex: Multi-Agent Research Workflow with AgentWorkflow
https://docs.llamaindex.ai/en/stable/examples/agent/agent_workflow_multi/


My Github repo: LLMs-in-Finance: https://github.com/hananedupouy/LLMs-in-Finance

* https://github.com/hananedupouy/LLMs-in-Finance/tree/main/Agents